# LeetCode 738 — Monotone Increasing Digits bằng Python

Link đề: https://leetcode.com/problems/monotone-increasing-digits/ · Độ khó: **Medium** · Chủ đề: Greedy, Math

Skeleton LeetCode:

```python
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        ...
```

Ghi chú: "monotone increasing digits" nghĩa là các chữ số **không giảm** từ trái sang phải (`x <= y` với mọi cặp chữ số liền kề). Bài yêu cầu tìm **số lớn nhất `<= n`** có tính chất này. Thử lùi dần từ `n` thì quá chậm; lời giải đẹp là **tham lam theo chữ số**: tìm chỗ chữ số **giảm** đầu tiên, hạ một chữ số ở bên trái đi 1 và **đổ toàn `9`** vào phần đuôi.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `0 <= n <= 10^9`, trả về số **lớn nhất không vượt quá `n`** mà mọi cặp chữ số liền kề `x, y` thỏa `x <= y`.
2. Nắm các công cụ Python: chuyển số sang chuỗi `str(n)` / `list(str(n))`, chữ số dạng ký tự và so sánh ký tự, `int(...)`, `''.join`, `zip(s, s[1:])`, `all`, `divmod` và lũy thừa của 10, `bisect`, `itertools.combinations_with_replacement`.
3. Đi từ **lùi từng đơn vị** → **quét từ phải sang trái và đánh dấu** → **quét trái sang phải tới chỗ giảm đầu tiên rồi lùi lại** → **liệt kê vị trí hạ chữ số** → **phép trừ số học lặp**.
4. Hiểu mấu chốt: gọi `i` là chữ số đầu tiên có `d[i] > d[i+1]`; đáp án là hạ **chữ số đầu tiên của cụm chữ số bằng nhau chứa `i`** đi 1 rồi điền toàn `9` phía sau.
5. Tránh các lỗi kinh điển: chỉ hạ một chữ số mà không xử lý cụm chữ số bằng nhau (`332 → 329` sai, đúng là `299`), dùng `>=` thay `>` (coi chữ số bằng nhau là vi phạm), quên điền `9`, trả về chuỗi `"09"` thay vì số `9`, và lùi từng đơn vị (quá chậm).

## 2. Đề bài & ví dụ

Một số nguyên có **các chữ số tăng dần theo nghĩa không giảm** (monotone increasing digits) khi và chỉ khi mỗi cặp chữ số liền kề `x` và `y` đều thỏa `x <= y`.

Cho số nguyên `n`, trả về **số lớn nhất nhỏ hơn hoặc bằng `n`** có các chữ số tăng dần theo nghĩa trên.

**Ví dụ 1**

- Input: `n = 10`
- Output: `9`

**Ví dụ 2**

- Input: `n = 1234`
- Output: `1234`

**Ví dụ 3**

- Input: `n = 332`
- Output: `299`

**Ràng buộc:**

- `0 <= n <= 10^9`

Chú ý: số `1234` đã tăng dần nên đáp án là chính nó. Với `10`, cặp `1, 0` vi phạm (`1 > 0`), số lớn nhất `<= 10` thỏa là `9`. Với `332` thì `3, 3, 2` vi phạm ở cặp `3, 2`; thử `331, 330, ..., 300` đều có cặp `3, x` với `x < 3` nên vi phạm; kế đó `299` (`2 <= 9 <= 9`) thỏa.

In [1]:
from typing import List
from bisect import bisect_right, bisect_left
from itertools import combinations_with_replacement
import random
import time

examples = [
    (10, 9),
    (1234, 1234),
    (332, 299),
]
for n, expected in examples:
    s = str(n)
    print("n =", n, "| chữ số:", list(s), "| kỳ vọng:", expected)

n = 10 | chữ số: ['1', '0'] | kỳ vọng: 9
n = 1234 | chữ số: ['1', '2', '3', '4'] | kỳ vọng: 1234
n = 332 | chữ số: ['3', '3', '2'] | kỳ vọng: 299


## 3. Kiến thức nền

### 3.1. Chữ số của một số: `str(n)` và `list(str(n))`

`str(n)` đổi số nguyên thành chuỗi các ký tự chữ số, ví dụ `str(332) == "332"`. Chuỗi là bất biến (immutable) nên muốn **sửa từng chữ số** ta dùng `list(str(n))` (danh sách ký tự), sửa xong nối lại bằng `''.join(...)` rồi `int(...)`.

In [2]:
n = 332
s = str(n)
print(s, type(s), list(s))
digits = list(s)
digits[0] = "2"                 # sửa được trên list
print(digits, "".join(digits), int("".join(digits)))

332 <class 'str'> ['3', '3', '2']
['2', '3', '2'] 232 232


### 3.2. So sánh chữ số dạng ký tự

Ký tự chữ số so sánh theo mã, và **thứ tự của `'0' < '1' < ... < '9'` trùng với thứ tự của các số**. Nên `"3" > "2"` đúng và có thể so sánh trực tiếp `s[i-1] > s[i]` mà không cần đổi sang số. Muốn **trừ 1 một chữ số** ký tự: `str(int(c) - 1)` hoặc `chr(ord(c) - 1)`.

In [3]:
print("3" > "2", "9" > "10"[0], "0" < "5")
c = "7"
print(str(int(c) - 1), chr(ord(c) - 1))
print(all(a <= b for a, b in zip("1234", "234")), all(a <= b for a, b in zip("1224"[:-1], "1224"[1:])))

True True True
6 6
True True


### 3.3. Kiểm tra "các chữ số không giảm": `zip(s, s[1:])`

`zip(s, s[1:])` ghép mỗi chữ số với chữ số kế tiếp: `("3","3"), ("3","2")` cho `"332"`. Một số tăng dần (không giảm) khi **mọi** cặp thỏa `a <= b`: `all(a <= b for a, b in zip(s, s[1:]))`. Chuỗi 1 ký tự (không có cặp) luôn thỏa.

In [4]:
def is_monotone(n: int) -> bool:
    s = str(n)
    return all(a <= b for a, b in zip(s, s[1:]))

for n in (0, 7, 1234, 1224, 10, 332, 11, 100):
    print(n, is_monotone(n))

0 True
7 True
1234 True
1224 True
10 False
332 False
11 True
100 False


### 3.4. Lũy thừa của 10, `divmod` và `//`, `%`

`n // 10**m` bỏ `m` chữ số cuối; `n % 10**m` giữ `m` chữ số cuối. Ví dụ `332 // 10 = 33`, `332 % 10 = 2`. Phép `(n // 10**m) * 10**m - 1` đưa `n` về số **tròn trăm/nghìn** rồi trừ 1, tức **hạ chữ số rồi điền `9`** phía sau: `(332 // 10) * 10 - 1 = 329`.

In [5]:
n = 332
print(n // 10, n % 10, divmod(n, 10))
print((n // 10) * 10 - 1, (329 // 100) * 100 - 1)
print([10 ** m for m in range(5)])

33 2 (33, 2)
329 299
[1, 10, 100, 1000, 10000]


### 3.5. `bisect` — tìm trong danh sách đã sắp xếp

`bisect_right(a, x)` cho chỉ số `i` sao cho mọi phần tử `a[:i] <= x`; vậy `a[bisect_right(a, x) - 1]` là phần tử **lớn nhất `<= x`**. Ta dùng để làm **oracle**: liệt kê trước mọi số tăng dần rồi tra.

In [6]:
a = [1, 3, 5, 8, 13]
for x in (0, 1, 4, 13, 20):
    i = bisect_right(a, x)
    print(f"x={x}: lớn nhất <= x là", a[i - 1] if i else None)

x=0: lớn nhất <= x là None
x=1: lớn nhất <= x là 1
x=4: lớn nhất <= x là 3
x=13: lớn nhất <= x là 13
x=20: lớn nhất <= x là 13


### 3.6. `combinations_with_replacement`

`combinations_with_replacement("0123456789", L)` sinh mọi bộ `L` chữ số **không giảm** (chọn có lặp, giữ thứ tự tăng). Đây **chính xác** là tập các số có `L` chữ số (cho phép số 0 đứng đầu) thỏa tính chất đề bài. Ta dùng nó làm nguồn dữ liệu độc lập cho oracle.

In [7]:
combos = list(combinations_with_replacement("0123", 3))
print(len(combos), ["".join(c) for c in combos][:8])
# số bộ L chữ số không giảm từ 10 chữ số = C(L + 9, 9)
from math import comb
print(comb(3 + 3, 3), len(combos))

20 ['000', '001', '002', '003', '011', '012', '013', '022']
20 20


## 4. Ý tưởng nền tảng / trực giác

Hãy tự làm tay ví dụ 3: `n = 332`. Số `332` có `3, 3, 2`: cặp `3, 2` vi phạm (`3 > 2`).

Thử **lùi từng bước** `331, 330, 329, 328, ...`: `331` (`3 > 1`) vi phạm, `330` vi phạm, `329` (`3 > 2`) vi phạm... cho đến `299`. Các số từ `300` đến `332` đều có chữ số đầu là `3` và chữ số kế tiếp `0..3`; chỉ `333` mới thỏa nhưng `333 > 332`. Khi chữ số hàng trăm là `3`, hàng chục phải `>= 3` mà `n` chỉ cho phép `<= 3`; hàng chục `= 3` thì hàng đơn vị phải `>= 3` nhưng `n` cho `<= 2`: mâu thuẫn. Vậy **bắt buộc hạ chữ số hàng trăm** xuống `2`, và khi đã nhỏ hơn `n` ở hàng trăm, các hàng sau **tự do**: chọn lớn nhất, tức `9`, `9`: `299`.

Tổng quát hóa: nếu số đã tăng dần thì đáp án là chính nó. Nếu không, gọi `i` là vị trí **đầu tiên** có `d[i] > d[i+1]`. Mọi số thỏa mà `<= n` hoặc giữ nguyên tiền tố `d[0..i+1]` (vi phạm, không được), hoặc **nhỏ hơn `n` tại một chữ số `k <= i`**. Ta muốn `k` càng **lớn** càng tốt (giữ tiền tố dài hơn thì số lớn hơn), và các chữ số sau `k` điền `9`. Nhưng hạ `d[k]` xuống `d[k] - 1` phải còn `>= d[k-1]`: chỉ được khi `d[k-1] < d[k]` (hoặc `k = 0`). Trong cụm các chữ số bằng nhau liên tiếp kết thúc ở `i`, chỉ **chữ số đầu cụm** có `d[k-1] < d[k]`: đó là `k` cần tìm.

Hãy kiểm chứng bằng cách liệt kê thật nhỏ:

In [8]:
def brute_largest(n):
    while not is_monotone(n):
        n -= 1
    return n

for n in (10, 1234, 332, 120, 668841, 7000, 98):
    s = str(n)
    i = next((i for i in range(len(s) - 1) if s[i] > s[i + 1]), None)
    if i is None:
        print(f"n={n}: đã tăng dần -> {n}")
        continue
    j = i
    while j > 0 and s[j - 1] == s[i]:
        j -= 1                           # lùi về đầu cụm chữ số bằng nhau
    cand = int(s[:j] + str(int(s[j]) - 1) + "9" * (len(s) - j - 1))
    print(f"n={n}: chỗ giảm đầu tiên i={i}, đầu cụm j={j}, ứng viên={cand}, vét cạn={brute_largest(n)}")
    assert cand == brute_largest(n)

n=10: chỗ giảm đầu tiên i=0, đầu cụm j=0, ứng viên=9, vét cạn=9
n=1234: đã tăng dần -> 1234
n=332: chỗ giảm đầu tiên i=1, đầu cụm j=0, ứng viên=299, vét cạn=299
n=120: chỗ giảm đầu tiên i=1, đầu cụm j=1, ứng viên=119, vét cạn=119
n=668841: chỗ giảm đầu tiên i=3, đầu cụm j=2, ứng viên=667999, vét cạn=667999
n=7000: chỗ giảm đầu tiên i=0, đầu cụm j=0, ứng viên=6999, vét cạn=6999
n=98: chỗ giảm đầu tiên i=0, đầu cụm j=0, ứng viên=89, vét cạn=89


## 5. Cách 1 — Brute Force: lùi từng đơn vị

**Ý tưởng.** Bắt đầu từ `n`, kiểm tra các chữ số có tăng dần không; nếu không thì giảm `n` đi `1` và kiểm tra tiếp, cho đến khi gặp số thỏa mãn. Vì `0` luôn thỏa nên vòng lặp luôn dừng.

In [9]:
def monotone_brute(n):
    while not is_monotone(n):
        n -= 1
    return n


for n, expected in examples:
    got = monotone_brute(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

10 -> 9 (kỳ vọng 9 )
1234 -> 1234 (kỳ vọng 1234 )
332 -> 299 (kỳ vọng 299 )


**Độ phức tạp:** Time `O((n - ans) · log n)`: số lần lùi bằng `n - đáp án`, mỗi lần kiểm tra `O(log n)` chữ số. Trong trường hợp xấu `n - đáp án` rất lớn (ví dụ `n = 332000000`, đáp án `299999999`: phải lùi hơn `32` triệu lần). Space `O(log n)`.

**Nút thắt:** mỗi bước lùi chỉ giảm `1`, trong khi ta biết ngay chỗ nào vi phạm. Ta có thể **nhảy thẳng** tới số tăng dần lớn nhất mà không duyệt các số trung gian: đó là bước đột phá ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi các chữ số của `n` là `d[0], d[1], ..., d[L-1]` (từ trái sang phải).

**Trường hợp 1.** Không có `i` nào có `d[i] > d[i+1]`: `n` đã tăng dần → đáp án là `n`.

**Trường hợp 2.** Có chỗ giảm. Gọi `i` là chỉ số **nhỏ nhất** có `d[i] > d[i+1]`. Khi đó `d[0] <= d[1] <= ... <= d[i]` (tiền tố tăng dần). Gọi `j` là chỉ số **nhỏ nhất** sao cho `d[j] = d[j+1] = ... = d[i]` (đầu cụm chữ số bằng `d[i]`). Đáp án là:

```
d[0..j-1]  ,  d[j] - 1  ,  9 9 9 ... 9   (L - j - 1 chữ số 9)
```

(nếu `d[0] - 1 = 0` ở đầu thì chuyển sang số nguyên sẽ tự bỏ số `0` thừa: ví dụ `10 → "09" → 9`.)

**Cách hiện thực gọn: quét từ phải sang trái.** Đặt `mark = L` (vị trí bắt đầu điền `9`). Duyệt `i` từ `L-1` về `1`: nếu `d[i-1] > d[i]` thì **hạ `d[i-1]` đi 1** và đặt `mark = i`. Hạ `d[i-1]` có thể làm `d[i-2] > d[i-1]` mới (cụm chữ số bằng nhau), nhưng vòng lặp đi tiếp sang trái sẽ **phát hiện và sửa**. Cuối cùng điền `9` từ `mark` đến hết. Cả quá trình `O(L)`.

Hai cách hiện thực khác: **quét trái-phải rồi lùi** (doocs) và **phép trừ số học lặp** (`n = (n // 10^m) * 10^m - 1` tại chỗ giảm phải nhất).

## 7. Các cách tiếp theo

Phần này có thêm 4 cách nữa (tổng 5 cách): quét phải-trái và đánh dấu (cách nên nộp), quét trái-phải rồi lùi lại, liệt kê vị trí hạ chữ số, và phép trừ số học lặp.

### 7.1. Cách 2 — Quét từ phải sang trái và đánh dấu vị trí điền `9`

**Giải thích từng đoạn.**

1. `s = list(str(n))`, `mark = len(s)` (chưa có chỗ điền `9`).
2. `for i in range(len(s) - 1, 0, -1)`: so `s[i-1]` với `s[i]`. Nếu `s[i-1] > s[i]`: hạ `s[i-1]` đi 1, và `mark = i` (từ `i` trở đi sẽ là `9`).
3. Sau vòng lặp: `for k in range(mark, len(s)): s[k] = '9'`. Trả `int(''.join(s))`.

Vì quét từ phải sang trái, mỗi lần hạ chữ số `s[i-1]` đã làm cặp `(s[i-2], s[i-1])` mới có thể vi phạm, và bước kế tiếp xử lý nó.

In [10]:
def monotone_right_to_left(n):
    s = list(str(n))
    mark = len(s)                              # từ mark trở đi sẽ điền '9'
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)  # hạ chữ số bên trái đi 1
            mark = i
    for k in range(mark, len(s)):
        s[k] = "9"
    return int("".join(s))


for n, expected in examples:
    got = monotone_right_to_left(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

10 -> 9 (kỳ vọng 9 )
1234 -> 1234 (kỳ vọng 1234 )
332 -> 299 (kỳ vọng 299 )


In [11]:
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        s = list(str(n))
        mark = len(s)
        for i in range(len(s) - 1, 0, -1):
            if s[i - 1] > s[i]:
                s[i - 1] = str(int(s[i - 1]) - 1)
                mark = i
        for k in range(mark, len(s)):
            s[k] = "9"
        return int("".join(s))

print([Solution().monotoneIncreasingDigits(n) for n, _ in examples])

[9, 1234, 299]


**Độ phức tạp:** Time `O(L)` với `L = log10(n)` là số chữ số (≤ 10). Space `O(L)` cho danh sách chữ số.

### 7.2. Cách 3 — Quét trái sang phải tới chỗ giảm đầu tiên, rồi lùi lại

**Giải thích từng đoạn.** Đây là cách của lời giải tham khảo (doocs).

1. Tìm `i` nhỏ nhất sao cho `s[i-1] > s[i]` (bằng cách đi tới trong khi `s[i-1] <= s[i]`). Nếu không có (`i == L`) thì trả `n`.
2. **Lùi lại:** trong khi `i > 0` và `s[i-1] > s[i]`: hạ `s[i-1]` đi 1 rồi `i -= 1`. Sau khi hạ, cặp bên trái `(s[i-1], s[i])` (với `i` mới) có thể lại vi phạm nếu `s[i-1]` từng **bằng** chữ số vừa hạ (cụm chữ số bằng nhau); vòng lặp tiếp tục lùi cho tới khi cặp không còn vi phạm (đầu cụm).
3. Điền `9` từ vị trí `i + 1` đến hết.

In [12]:
def monotone_forward_then_back(n):
    s = list(str(n))
    i = 1
    while i < len(s) and s[i - 1] <= s[i]:
        i += 1                                  # i: chỗ giảm đầu tiên (hoặc len(s))
    if i < len(s):
        while i and s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)   # hạ và lùi sang trái
            i -= 1
        i += 1
        while i < len(s):
            s[i] = "9"
            i += 1
    return int("".join(s))


for n, expected in examples:
    got = monotone_forward_then_back(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

10 -> 9 (kỳ vọng 9 )
1234 -> 1234 (kỳ vọng 1234 )
332 -> 299 (kỳ vọng 299 )


In [13]:
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        s = list(str(n))
        i = 1
        while i < len(s) and s[i - 1] <= s[i]:
            i += 1
        if i < len(s):
            while i and s[i - 1] > s[i]:
                s[i - 1] = str(int(s[i - 1]) - 1)
                i -= 1
            i += 1
            while i < len(s):
                s[i] = "9"
                i += 1
        return int("".join(s))

print([Solution().monotoneIncreasingDigits(n) for n, _ in examples])

[9, 1234, 299]


**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

*Ghi chú:* sau khi hạ `s[i-1]` và giảm `i`, điều kiện lặp `s[i - 1] > s[i]` so cặp mới `(s[i-1], s[i])`, trong đó `s[i]` chính là chữ số vừa bị hạ. Cặp này vi phạm khi chữ số bên trái **bằng** chữ số ban đầu trước khi hạ (cụm bằng nhau). Mỗi lần lặp lùi một bước qua cụm. Cuối cùng `i` dừng ở chữ số cuối cùng bị hạ (đầu cụm); sau đó `i += 1` là vị trí bắt đầu điền `9`.

### 7.3. Cách 4 — Liệt kê vị trí "bắt đầu nhỏ hơn `n`" rồi lấy lớn nhất

**Giải thích từng đoạn.** Mọi số `m <= n` hoặc bằng `n`, hoặc **giống `n` ở một tiền tố rồi nhỏ hơn ở chữ số thứ `k`**. Với mỗi `k`, số **lớn nhất** thuộc dạng đó là `d[0..k-1] + (d[k] - 1) + 9...9`. Ta dựng mọi ứng viên (cho `k = 0..L-1` với `d[k] > 0`), **giữ lại các ứng viên tăng dần**, cộng thêm `n` nếu `n` tăng dần, và lấy lớn nhất. Không cần chứng minh tinh tế về "cụm bằng nhau": phép kiểm tra tăng dần tự loại ứng viên sai. Độ phức tạp `O(L^2)` nhưng `L ≤ 10` nên không đáng kể.

In [14]:
def monotone_enumerate(n):
    s = str(n)
    best = n if is_monotone(n) else -1
    for k in range(len(s)):
        if s[k] == "0":
            continue                                  # không hạ được chữ số 0
        cand = int(s[:k] + str(int(s[k]) - 1) + "9" * (len(s) - k - 1))
        if is_monotone(cand):
            best = max(best, cand)
    return best


for n, expected in examples:
    got = monotone_enumerate(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

10 -> 9 (kỳ vọng 9 )
1234 -> 1234 (kỳ vọng 1234 )
332 -> 299 (kỳ vọng 299 )


In [15]:
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        def ok(x):
            t = str(x)
            return all(a <= b for a, b in zip(t, t[1:]))

        s = str(n)
        best = n if ok(n) else -1
        for k in range(len(s)):
            if s[k] == "0":
                continue
            cand = int(s[:k] + str(int(s[k]) - 1) + "9" * (len(s) - k - 1))
            if ok(cand):
                best = max(best, cand)
        return best

print([Solution().monotoneIncreasingDigits(n) for n, _ in examples])

[9, 1234, 299]


**Độ phức tạp:** Time `O(L^2)` (`L` ứng viên, mỗi ứng viên dựng và kiểm tra `O(L)`). Space `O(L)`. Cách này nói rõ **vì sao** đáp án có dạng "tiền tố + hạ 1 + toàn 9", và là một oracle tốt nhưng độc lập với cách quét.

### 7.4. Cách 5 — Phép trừ số học lặp (không dùng chuỗi để sửa)

**Giải thích từng đoạn.** Lặp: tìm chỗ giảm **phải nhất** `j` (`d[j] > d[j+1]` với `j` lớn nhất); đặt `m = L - 1 - j` (số chữ số sau `j`); cập nhật `n = (n // 10**m) * 10**m - 1`. Phép này giữ nguyên tiền tố `d[0..j]`, làm tròn phần còn lại về `0...0` rồi trừ 1: tức hạ `d[j]` đi 1 và điền `9` cho `m` chữ số sau. Mọi số bị bỏ qua đều có chỗ giảm tại `(j, j+1)` nên không thỏa. Lặp cho tới khi `n` tăng dần. Mỗi vòng `n` ít nhất mất một chữ số vi phạm; tối đa `L` vòng.

In [16]:
def monotone_arithmetic(n):
    while True:
        s = str(n)
        j = -1
        for t in range(len(s) - 1):
            if s[t] > s[t + 1]:
                j = t                              # chỗ giảm phải nhất
        if j == -1:
            return n
        m = len(s) - 1 - j
        n = (n // 10 ** m) * 10 ** m - 1


for n, expected in examples:
    got = monotone_arithmetic(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

10 -> 9 (kỳ vọng 9 )
1234 -> 1234 (kỳ vọng 1234 )
332 -> 299 (kỳ vọng 299 )


In [17]:
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        while True:
            s = str(n)
            j = -1
            for t in range(len(s) - 1):
                if s[t] > s[t + 1]:
                    j = t
            if j == -1:
                return n
            m = len(s) - 1 - j
            n = (n // 10 ** m) * 10 ** m - 1

print([Solution().monotoneIncreasingDigits(n) for n, _ in examples])

[9, 1234, 299]


**Độ phức tạp:** Time `O(L^2)` (tối đa `L` vòng, mỗi vòng `O(L)`). Space `O(L)`. Cách này gần với "tính bằng toán" và cho thấy mỗi bước đều **bỏ qua đúng các số không thỏa**.

## 8. TRACE từng bước

Trace cách 2 (phải → trái) với `n = 332`, kỳ vọng `299`. `s = ['3','3','2']`, `mark = 3`.

| bước | `i` | so sánh `s[i-1]` với `s[i]` | hành động | `s` sau | `mark` |
|---|---|---|---|---|---|
| 1 | 2 | `s[1]='3' > s[2]='2'` | hạ `s[1]` → `'2'`, `mark = 2` | `['3','2','2']` | 2 |
| 2 | 1 | `s[0]='3' > s[1]='2'` | hạ `s[0]` → `'2'`, `mark = 1` | `['2','2','2']` | 1 |
| điền 9 | | từ `mark = 1` đến hết | `s[1] = s[2] = '9'` | `['2','9','9']` | |

Kết quả `299`. Chú ý bước 2: sau khi hạ `s[1]` thành `'2'`, cặp `(s[0], s[1]) = ('3','2')` mới vi phạm: vòng lặp đi sang trái đã gặp và sửa ngay.

In [18]:
def trace_right_to_left(n):
    s = list(str(n))
    mark = len(s)
    print(f"n={n}, s={s}, mark={mark}")
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] > s[i]:
            old = s[i - 1]
            s[i - 1] = str(int(old) - 1)
            mark = i
            print(f"  i={i}: s[{i - 1}]={old!r} > s[{i}]={s[i]!r} -> hạ s[{i - 1}] thành {s[i - 1]!r}, mark={mark}; s={s}")
        else:
            print(f"  i={i}: s[{i - 1}]={s[i - 1]!r} <= s[{i}]={s[i]!r} -> giữ nguyên; s={s}")
    for k in range(mark, len(s)):
        s[k] = "9"
    print(f"  điền 9 từ vị trí {mark}: s={s}")
    ans = int("".join(s))
    print("  đáp án =", ans)
    return ans

assert trace_right_to_left(332) == 299

n=332, s=['3', '3', '2'], mark=3
  i=2: s[1]='3' > s[2]='2' -> hạ s[1] thành '2', mark=2; s=['3', '2', '2']
  i=1: s[0]='3' > s[1]='2' -> hạ s[0] thành '2', mark=1; s=['2', '2', '2']
  điền 9 từ vị trí 1: s=['2', '9', '9']
  đáp án = 299


Trace `n = 10`: `s = ['1','0']`, `mark = 2`. `i = 1`: `'1' > '0'` → hạ `s[0]` thành `'0'`, `mark = 1`. Điền `9` từ vị trí `1`: `['0','9']` → `int("09") = 9`. Chú ý số `0` đứng đầu được `int` bỏ đi.

Trace `n = 1234`: mọi cặp `<=`, không có thay đổi, `mark = 4`, không điền `9`, đáp án `1234`.

In [19]:
assert trace_right_to_left(10) == 9
print()
assert trace_right_to_left(1234) == 1234
print()
assert trace_right_to_left(668841) == 667999

n=10, s=['1', '0'], mark=2
  i=1: s[0]='1' > s[1]='0' -> hạ s[0] thành '0', mark=1; s=['0', '0']
  điền 9 từ vị trí 1: s=['0', '9']
  đáp án = 9

n=1234, s=['1', '2', '3', '4'], mark=4
  i=3: s[2]='3' <= s[3]='4' -> giữ nguyên; s=['1', '2', '3', '4']
  i=2: s[1]='2' <= s[2]='3' -> giữ nguyên; s=['1', '2', '3', '4']
  i=1: s[0]='1' <= s[1]='2' -> giữ nguyên; s=['1', '2', '3', '4']
  điền 9 từ vị trí 4: s=['1', '2', '3', '4']
  đáp án = 1234

n=668841, s=['6', '6', '8', '8', '4', '1'], mark=6
  i=5: s[4]='4' > s[5]='1' -> hạ s[4] thành '3', mark=5; s=['6', '6', '8', '8', '3', '1']
  i=4: s[3]='8' > s[4]='3' -> hạ s[3] thành '7', mark=4; s=['6', '6', '8', '7', '3', '1']
  i=3: s[2]='8' > s[3]='7' -> hạ s[2] thành '7', mark=3; s=['6', '6', '7', '7', '3', '1']
  i=2: s[1]='6' <= s[2]='7' -> giữ nguyên; s=['6', '6', '7', '7', '3', '1']
  i=1: s[0]='6' <= s[1]='6' -> giữ nguyên; s=['6', '6', '7', '7', '3', '1']
  điền 9 từ vị trí 3: s=['6', '6', '7', '9', '9', '9']
  đáp án = 667999


Trace cách 3 (trái → phải, rồi lùi) cho `n = 332` để thấy sự khác nhau về trình tự:

| giai đoạn | `i` | `s` | ghi chú |
|---|---|---|---|
| tiến | 1 → 2 | `['3','3','2']` | `s[0]<=s[1]` nên `i=2`; `s[1]>s[2]` dừng ở `i = 2` |
| lùi | 2 | hạ `s[1]` → `['3','2','2']`, `i = 1` | `s[0]='3' > s[1]='2'` tiếp tục |
| lùi | 1 | hạ `s[0]` → `['2','2','2']`, `i = 0` | `i = 0` dừng |
| điền 9 | `i += 1 = 1` | `['2','9','9']` | từ vị trí 1 đến hết |

In [20]:
def trace_forward_back(n):
    s = list(str(n))
    i = 1
    while i < len(s) and s[i - 1] <= s[i]:
        i += 1
    print(f"n={n}: chỗ giảm đầu tiên ở i={i} (len={len(s)})")
    if i < len(s):
        while i and s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)
            i -= 1
            print(f"  lùi: hạ s[{i}] -> i={i}, s={s}")
        i += 1
        while i < len(s):
            s[i] = "9"
            i += 1
        print(f"  điền 9: s={s}")
    ans = int("".join(s))
    print("  đáp án =", ans)
    return ans

assert trace_forward_back(332) == 299
assert trace_forward_back(1234) == 1234

n=332: chỗ giảm đầu tiên ở i=2 (len=3)
  lùi: hạ s[1] -> i=1, s=['3', '2', '2']
  lùi: hạ s[0] -> i=0, s=['2', '2', '2']
  điền 9: s=['2', '9', '9']
  đáp án = 299
n=1234: chỗ giảm đầu tiên ở i=4 (len=4)
  đáp án = 1234


## 9. Code nộp LeetCode

Lời giải quét phải → trái (Cách 2):

In [21]:
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        s = list(str(n))
        mark = len(s)                                   # từ mark trở đi điền '9'
        for i in range(len(s) - 1, 0, -1):
            if s[i - 1] > s[i]:
                s[i - 1] = str(int(s[i - 1]) - 1)       # hạ chữ số bên trái đi 1
                mark = i
        for k in range(mark, len(s)):
            s[k] = "9"
        return int("".join(s))


sol = Solution()
for n, expected in examples:
    got = sol.monotoneIncreasingDigits(n)
    print(got)
    assert got == expected

9
1234
299


## 10. Vì sao đúng

Ký hiệu `d[0..L-1]` là các chữ số của `n`.

**Bổ đề 1 — dạng của số tối ưu.** Nếu `n` không tăng dần, giả sử `m` là đáp án (`m < n`, `m` tăng dần, lớn nhất). Viết `m` và `n` với cùng số chữ số `L` (đệm `0` ở đầu nếu cần). Tồn tại vị trí `k` đầu tiên mà `m[k] != n[k]`, và `m[k] < n[k]` (vì `m < n`). Với `k` cố định, trong các `m` có cùng tiền tố `d[0..k-1]` và `m[k] = c < d[k]`, số **lớn nhất** có `c = d[k] - 1` rồi toàn `9` phía sau (cách điền lớn nhất, và dãy `c, 9, ..., 9` không giảm miễn là tiền tố hợp lệ).

**Bổ đề 2 — điều kiện cho `k`.** Để ứng viên `d[0..k-1] , d[k]-1 , 9...9` tăng dần cần: (a) tiền tố `d[0..k-1]` tăng dần, và (b) `d[k-1] <= d[k] - 1`, tức `d[k-1] < d[k]` (hoặc `k = 0`). Gọi `i` là chỗ giảm đầu tiên. Nếu `k >= i + 2` thì tiền tố chứa cặp vi phạm `(d[i], d[i+1])`, loại. Nếu `k = i + 1` thì `d[k-1] = d[i] > d[i+1] = d[k]`, vi phạm (b), loại. Vậy chỉ còn `k <= i` với `d[k-1] < d[k]` hoặc `k = 0`.

**Bổ đề 3 — chọn `k` lớn nhất.** Ứng viên có `k` lớn hơn giữ tiền tố của `n` dài hơn nên **lớn hơn** (tại vị trí `k_nhỏ`, ứng viên `k_lớn` giữ `d[k_nhỏ]` còn ứng viên `k_nhỏ` có `d[k_nhỏ] - 1`). Trong các `k <= i` thỏa `d[k-1] < d[k]`, `k` lớn nhất là **đầu cụm các chữ số bằng `d[i]`** (vì tiền tố `d[0..i]` không giảm, các chữ số bằng nhau liên tiếp trước `i` đều có `d[k-1] = d[k]`, không thỏa, đến đầu cụm thì `d[j-1] < d[j]` hoặc `j = 0`).

**Bổ đề 4 — các cách hiện thực.** Cách 2 (phải → trái) hạ `d[i-1]` tại mọi cặp giảm gặp trên đường đi sang trái; sau khi hạ, cặp bên trái có thể giảm và tiếp tục hạ: kết quả là hạ đúng chữ số **đầu cụm** (các chữ số trong cụm giữa bị hạ rồi bị ghi đè bởi `9`), và `mark` kết thúc ở vị trí ngay sau đầu cụm. Cách 3 làm cùng việc theo hướng ngược. Cách 4 dựng thẳng mọi ứng viên `k`; cách 5 lặp phép trừ.

**Kết luận.** Đáp án là chính `n` nếu `n` tăng dần, ngược lại là `d[0..j-1] , d[j] - 1 , 9...9` với `j` là đầu cụm chứa chỗ giảm đầu tiên. Số `0` đầu (nếu `j = 0` và `d[0] = 1`) được `int` bỏ đi.

In [22]:
# Kiểm chứng Bổ đề 1-3 bằng thực nghiệm với oracle vét cạn trên một dải rộng
best_dp = [0] * 200_001
for x in range(1, 200_001):
    best_dp[x] = x if is_monotone(x) else best_dp[x - 1]
for x in range(0, 200_001):
    assert monotone_right_to_left(x) == best_dp[x], x
print("Khớp DP vét cạn với mọi n trong [0, 200000]")

Khớp DP vét cạn với mọi n trong [0, 200000]


## 11. Time / Space Complexity

Đặt `L` là số chữ số của `n` (`L ≤ 10` vì `n ≤ 10^9`).

| Cách | Time | Space |
|---|---|---|
| 1. Lùi từng đơn vị | `O((n - ans) · L)` | `O(L)` |
| 2. Quét phải → trái, đánh dấu | `O(L)` | `O(L)` |
| 3. Quét trái → phải rồi lùi | `O(L)` | `O(L)` |
| 4. Liệt kê vị trí hạ | `O(L^2)` | `O(L)` |
| 5. Phép trừ số học lặp | `O(L^2)` | `O(L)` |

Với `L ≤ 10` mọi cách trừ cách 1 đều tức thời. Cách 1 có thể chậm vài chục triệu lần lặp trên đầu vào xấu; ta đo số bước thay vì chạy thật.

In [23]:
for n in (332_000_000, 999_999_990, 1_000_000_000, 100_000_000, 987_654_321):
    ans = monotone_right_to_left(n)
    print(f"n={n:>13,} -> {ans:>13,} | cách 1 phải lùi {n - ans:>11,} lần")
t0 = time.perf_counter()
for _ in range(20_000):
    monotone_right_to_left(987_654_321)
print("20.000 lần cách 2 chạy xong:", time.perf_counter() - t0 < 5)

n=  332,000,000 ->   299,999,999 | cách 1 phải lùi  32,000,001 lần
n=  999,999,990 ->   899,999,999 | cách 1 phải lùi  99,999,991 lần
n=1,000,000,000 ->   999,999,999 | cách 1 phải lùi           1 lần
n=  100,000,000 ->    99,999,999 | cách 1 phải lùi           1 lần
n=  987,654,321 ->   899,999,999 | cách 1 phải lùi  87,654,322 lần
20.000 lần cách 2 chạy xong: True


## 12. Bộ test

Chia 4 nhóm: (a) ví dụ đề và edge case có đáp án tính tay; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu mọi cách với **oracle độc lập** (liệt kê mọi số tăng dần bằng `combinations_with_replacement`, rồi `bisect`); (d) test ngẫu nhiên có seed trên toàn miền `[0, 10^9]`.

### 12.1. Oracle độc lập và test cố định

Oracle: sinh **mọi** số có tối đa 10 chữ số mà chữ số không giảm (tổng `184755` số), sắp xếp; đáp án của `n` là phần tử lớn nhất `<= n` (tra bằng `bisect`). Không dùng bất kỳ phép "hạ rồi điền 9" nào.

In [24]:
monotone_numbers = sorted(
    int("".join(c))
    for L in range(1, 11)
    for c in combinations_with_replacement("0123456789", L)
)
monotone_numbers = sorted(set(monotone_numbers))
print("số lượng số tăng dần có <= 10 chữ số:", len(monotone_numbers))


def oracle(n):
    return monotone_numbers[bisect_right(monotone_numbers, n) - 1]


solvers = {
    "right_to_left": monotone_right_to_left,
    "forward_back": monotone_forward_then_back,
    "enumerate": monotone_enumerate,
    "arithmetic": monotone_arithmetic,
    "submit": lambda n: Solution().monotoneIncreasingDigits(n),
}

fixed_cases = [
    (10, 9), (1234, 1234), (332, 299),          # ví dụ đề
    (0, 0), (1, 1), (9, 9),                      # một chữ số
    (11, 11), (22, 22), (99, 99),                # chữ số bằng nhau: vẫn thỏa
    (100, 99), (120, 119), (98, 89), (321, 299), # hạ ở đầu cụm
    (1243, 1239), (7000, 6999), (668841, 667999),
    (1000000000, 999999999),                     # cận trên
    (999999999, 999999999),
    (123454321, 123449999),
    (233221, 229999),
]
for n, expected in fixed_cases:
    assert oracle(n) == expected, ("oracle", n, oracle(n), expected)
    assert monotone_brute(n) == expected if n < 2_000_000 else True
    for name, f in solvers.items():
        got = f(n)
        assert got == expected, (name, n, got, expected)
print("Test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải + oracle )")

số lượng số tăng dần có <= 10 chữ số: 92378
Test cố định: OK ( 20 bộ x 5 lời giải + oracle )


### 12.2. Phản ví dụ cho heuristic sai

- *"Chỉ hạ chữ số ngay trước chỗ giảm đầu tiên rồi điền `9`"* (không xử lý cụm chữ số bằng nhau): `332 → 329` nhưng `329` vẫn vi phạm (`3 > 2`); đáp án đúng `299`.
- *"Điền `9` ngay sau chỗ giảm mà không hạ"*: ví dụ `10 → 19` lớn hơn `n`.

In [25]:
def wrong_decrement_only_first(n):
    s = list(str(n))
    i = 1
    while i < len(s) and s[i - 1] <= s[i]:
        i += 1
    if i < len(s):
        s[i - 1] = str(int(s[i - 1]) - 1)           # SAI: không lùi qua cụm chữ số bằng nhau
        for k in range(i, len(s)):
            s[k] = "9"
    return int("".join(s))


def wrong_fill_nines_only(n):
    s = list(str(n))
    for i in range(1, len(s)):
        if s[i - 1] > s[i]:
            for k in range(i, len(s)):
                s[k] = "9"                           # SAI: không hạ chữ số bên trái
            break
    return int("".join(s))


for n in (332, 100, 668841):
    print(n, "| hạ-chỉ-lần-đầu:", wrong_decrement_only_first(n), "| đúng:", oracle(n))
print("điền 9 không hạ:", wrong_fill_nines_only(10), "> n = 10: vượt n")
assert wrong_decrement_only_first(332) == 329 and oracle(332) == 299
assert wrong_fill_nines_only(10) == 19 and oracle(10) == 9
assert not is_monotone(wrong_decrement_only_first(332))

332 | hạ-chỉ-lần-đầu: 329 | đúng: 299
100 | hạ-chỉ-lần-đầu: 99 | đúng: 99
668841 | hạ-chỉ-lần-đầu: 668799 | đúng: 667999
điền 9 không hạ: 19 > n = 10: vượt n


### 12.3. Đối chiếu oracle trên mọi `n` nhỏ và quanh các số tăng dần

Vét **mọi** `n` trong `[0, 60000]` (kể cả cách 1 vét cạn), và với `n` quanh các **biên**: mỗi số tăng dần `m` (lấy mẫu đều), thử `m - 1`, `m`, `m + 1`.

In [26]:
checked = 0
for n in range(0, 60_001):
    expected = oracle(n)
    for name, f in solvers.items():
        assert f(n) == expected, (name, n)
    assert monotone_brute(n) == expected, n
    checked += 1
print("Vét mọi n trong [0, 60000]:", checked, "giá trị, khớp oracle và vét cạn")

boundary = 0
for m in monotone_numbers[::37]:
    for n in (m - 1, m, m + 1):
        if 0 <= n <= 10 ** 9:
            expected = oracle(n)
            for name, f in solvers.items():
                assert f(n) == expected, (name, n)
            boundary += 1
print("Quanh các số tăng dần (mẫu):", boundary, "giá trị, khớp oracle")

Vét mọi n trong [0, 60000]: 60001 giá trị, khớp oracle và vét cạn
Quanh các số tăng dần (mẫu): 3944 giá trị, khớp oracle


### 12.4. Test ngẫu nhiên có seed trên toàn miền

`n` ngẫu nhiên trong `[0, 10^9]`, cùng một số kiểu có cấu trúc: số có nhiều chữ số bằng nhau, số có nhiều số `0`, số dạng `d...d x ...`.

In [27]:
rng = random.Random(738)
for _ in range(20_000):
    n = rng.randint(0, 10 ** 9)
    expected = oracle(n)
    for name, f in solvers.items():
        assert f(n) == expected, (name, n)
print("Ngẫu nhiên đều trên [0, 10^9] (seed 738): 20000 giá trị, khớp oracle")

rng = random.Random(2026)
for _ in range(20_000):
    L = rng.randint(1, 9)
    pool = rng.choice(["01", "09", "0129", "3", "123456789", "0123456789"])
    s = "".join(rng.choice(pool) for _ in range(L))
    n = int(s)
    expected = oracle(n)
    for name, f in solvers.items():
        assert f(n) == expected, (name, n)
print("Ngẫu nhiên có cấu trúc (seed 2026): 20000 giá trị, khớp oracle")

Ngẫu nhiên đều trên [0, 10^9] (seed 738): 20000 giá trị, khớp oracle


Ngẫu nhiên có cấu trúc (seed 2026): 20000 giá trị, khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ hạ chữ số ngay trước chỗ giảm (quên cụm chữ số bằng nhau)

Với `332`, chỗ giảm đầu tiên là `(3, 2)` ở vị trí `1-2`. Chỉ hạ `s[1]` thành `2` rồi điền `9` được `329`, nhưng cặp `(3, 2)` ở đầu lại vi phạm. Phải tiếp tục **đi sang trái** hạ cả `s[0]` (đầu cụm): `299`.

In [28]:
print("sai:", wrong_decrement_only_first(332), "| đúng:", Solution().monotoneIncreasingDigits(332))
print("329 có tăng dần không?", is_monotone(329))
assert wrong_decrement_only_first(332) == 329 and not is_monotone(329)
assert Solution().monotoneIncreasingDigits(332) == 299

sai: 329 | đúng: 299
329 có tăng dần không? False


### Lỗi 2 — Dùng `>=` thay vì `>` khi tìm chỗ vi phạm

Đề cho phép hai chữ số **bằng nhau** (`x <= y`). Nếu coi `s[i-1] >= s[i]` là vi phạm, số như `11`, `22`, `1224` bị "sửa" oan.

In [29]:
def wrong_ge(n):
    s = list(str(n))
    mark = len(s)
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] >= s[i]:                           # SAI: phải là >
            s[i - 1] = str(int(s[i - 1]) - 1)
            mark = i
    for k in range(mark, len(s)):
        s[k] = "9"
    return int("".join(s))

print("sai:", wrong_ge(11), wrong_ge(1224), "| đúng:", Solution().monotoneIncreasingDigits(11), Solution().monotoneIncreasingDigits(1224))
assert wrong_ge(11) != 11 and Solution().monotoneIncreasingDigits(11) == 11

sai: 9 999 | đúng: 11 1224


### Lỗi 3 — Quên điền `9` ở phần đuôi

Sau khi hạ chữ số, các chữ số phía sau **tự do**, nên phải điền `9` để được số lớn nhất. Nếu quên, `332` ra `222` thay vì `299`.

In [30]:
def wrong_no_nines(n):
    s = list(str(n))
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)
    return int("".join(s))                              # SAI: thiếu bước điền 9

print("sai:", wrong_no_nines(332), "| đúng:", Solution().monotoneIncreasingDigits(332))
assert wrong_no_nines(332) == 222 and Solution().monotoneIncreasingDigits(332) == 299

sai: 222 | đúng: 299


### Lỗi 4 — Trả về chuỗi hoặc để lại số `0` ở đầu

Với `n = 10`, quy trình cho `['0','9']`. Trả `"".join(s)` được chuỗi `"09"` (sai kiểu và sai giá trị); phải `int("".join(s))` để ra `9`.

In [31]:
def wrong_return_string(n):
    s = list(str(n))
    mark = len(s)
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)
            mark = i
    for k in range(mark, len(s)):
        s[k] = "9"
    return "".join(s)                                   # SAI: chuỗi, không phải int

print(repr(wrong_return_string(10)), "| đúng:", repr(Solution().monotoneIncreasingDigits(10)))
assert wrong_return_string(10) == "09" and Solution().monotoneIncreasingDigits(10) == 9

'09' | đúng: 9


### Lỗi 5 — Lùi từng đơn vị

Cách 1 đúng nhưng quá chậm với `n` lớn: `n = 332000000` phải lùi hơn 32 triệu lần (mục 11). Dù đề giới hạn `n ≤ 10^9` và Python chạy khoảng vài triệu phép kiểm tra mỗi giây, thời gian vẫn vượt xa giới hạn. Nên dùng cách tham lam `O(L)`.

In [32]:
n = 332_000_000
ans = Solution().monotoneIncreasingDigits(n)
print("đáp án:", ans, "| cách 1 phải lùi", n - ans, "lần")
assert n - ans == 32_000_001

đáp án: 299999999 | cách 1 phải lùi 32000001 lần


## 14. Mẹo & nhận dạng pattern

- **"Số lớn nhất `<= n` thỏa tính chất theo từng chữ số"** → làm việc trên **chuỗi chữ số**; thường là: giữ tiền tố dài nhất có thể, hạ một chữ số, rồi điền chữ số **tốt nhất** (ở đây là `9`) cho phần còn lại.
- **Tính chất liên quan tới cặp liền kề** → quét một lượt; khi sửa một chữ số hãy nghĩ **ảnh hưởng ngược sang trái** (cụm chữ số bằng nhau) → quét từ phải sang trái và dùng một biến `mark`.
- **Tham lam theo chữ số**: ưu tiên giữ tiền tố của `n` dài nhất; chỉ nhỏ hơn `n` tại vị trí muộn nhất có thể.
- **Số có `0` đứng đầu sau khi hạ** → chuyển sang `int`, không giữ chuỗi.
- **Oracle**: với tính chất kiểu "không giảm" có thể liệt kê mọi số hợp lệ bằng `combinations_with_replacement` rồi `bisect`.
- Các bài cùng họ: Remove K Digits (402), Maximum Swap (670), Next Greater Element III (556), Numbers With Same Consecutive Differences (967), Numbers At Most N Given Digit Set (902).

## 15. Biến thể

### 15.1. Số **nhỏ nhất** `>= n` có chữ số không giảm

Tìm chỗ giảm đầu tiên `i` (`d[i] > d[i+1]`); nếu không có, đáp án là `n`. Nếu có, giữ tiền tố `d[0..i]` rồi **điền `d[i]` vào mọi vị trí sau**: số nhỏ nhất có tiền tố đó mà không giảm (mỗi chữ số phải `>= d[i]`, nên chọn đúng `d[i]`).

In [33]:
def monotone_smallest_at_least(n):
    s = str(n)
    for i in range(len(s) - 1):
        if s[i] > s[i + 1]:
            return int(s[:i + 1] + s[i] * (len(s) - i - 1))
    return n

for n in (10, 1234, 332, 120, 987):
    print(n, "->", monotone_smallest_at_least(n))
for n in range(0, 20_000):
    i = bisect_left(monotone_numbers, n)
    assert monotone_smallest_at_least(n) == monotone_numbers[i], n
print("OK: khớp oracle trên [0, 20000)")

10 -> 11
1234 -> 1234
332 -> 333
120 -> 122
987 -> 999
OK: khớp oracle trên [0, 20000)


### 15.2. Đếm số lượng số tăng dần `<= n`

Với `n <= 10^9`, ta có thể đếm bằng oracle: `bisect_right(monotone_numbers, n)` (đã tính cả `0`). Cách toán học: số chữ số không giảm độ dài `L` là `C(L + 9, 9)` (chọn có lặp). Ví dụ có `C(12, 9) = 220` số tăng dần có 3 chữ số (kể cả số 0 đứng đầu).

In [34]:
from math import comb

def count_monotone_leq(n):
    return bisect_right(monotone_numbers, n)

print(count_monotone_leq(9), count_monotone_leq(99), count_monotone_leq(999))
print(comb(2 + 9, 9), comb(3 + 9, 9))
# số tăng dần <= 999 (gồm 0 và mọi số 1..3 chữ số): chính là các bộ 3 chữ số không giảm
assert count_monotone_leq(999) == comb(3 + 9, 9)
assert count_monotone_leq(99) == comb(2 + 9, 9)

10 55 220
55 220


### 15.3. Biến thể khác (chỉ mô tả)

- **Chữ số giảm dần (không tăng)**: số lớn nhất `<= n` có chữ số không tăng. Đây là bài tập 5 ở mục 17.
- **Chữ số tăng chặt** (`x < y`): tương tự nhưng không điền `9`; cần xử lý bằng dịch chỉ số.
- **`n` rất lớn (chuỗi dài hàng nghìn chữ số)**: cách 2 vẫn chạy `O(L)`, chỉ cần giữ `n` ở dạng chuỗi, không đổi sang `int`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lùi từng đơn vị | giảm `n` đến khi tăng dần | `O((n - ans) L)` | `O(L)` | chỉ làm oracle với `n` nhỏ |
| 2. Quét phải → trái | hạ `s[i-1]` khi `s[i-1] > s[i]`, đánh dấu `mark`, điền `9` | `O(L)` | `O(L)` | **cách nên nộp**: ngắn, ít lỗi |
| 3. Quét trái → phải rồi lùi | tìm chỗ giảm đầu tiên, lùi hạ cụm, điền `9` | `O(L)` | `O(L)` | lời giải tham khảo; giống cách 2 |
| 4. Liệt kê vị trí hạ | dựng ứng viên `tiền tố + (d[k]-1) + 99..9`, lấy lớn nhất hợp lệ | `O(L^2)` | `O(L)` | khi muốn thấy rõ lập luận; oracle phụ |
| 5. Phép trừ số học lặp | `n = (n // 10^m) * 10^m - 1` tại chỗ giảm phải nhất | `O(L^2)` | `O(L)` | khi muốn làm bằng toán, không sửa chuỗi |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(list(str(1204)))
print("3" > "2", "9" > "10", int("0099"))
print(all(a <= b for a, b in zip("1224", "224")))
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 233221`: tìm chỗ giảm đầu tiên `i`, đầu cụm chữ số bằng nhau `j`, rồi viết đáp án.

In [36]:
# Bài 2: điền kết quả tính tay
hand_first_drop = None      # chỉ số i đầu tiên có d[i] > d[i+1]
hand_run_start = None       # chỉ số j đầu cụm chữ số bằng d[i]
hand_answer = None          # đáp án

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách quét phải → trái.

In [37]:
def exercise_3_monotone(n: int) -> int:
    s = list(str(n))
    mark = len(s)
    # TODO 1: duyệt i từ len(s) - 1 về 1
    #     TODO 2: nếu s[i-1] > s[i]: hạ s[i-1] đi 1 và đặt mark = i
    # TODO 3: điền '9' cho mọi vị trí từ mark đến hết
    # TODO 4: trả về số nguyên (int) của chuỗi kết quả
    return None


# assert exercise_3_monotone(10) == 9
# assert exercise_3_monotone(1234) == 1234
# assert exercise_3_monotone(332) == 299

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 5, hãy tự viết `exercise_4_arithmetic(n)`: lặp tìm chỗ giảm phải nhất rồi cập nhật `n = (n // 10**m) * 10**m - 1`.

In [38]:
def exercise_4_arithmetic(n: int) -> int:
    # TODO
    pass


# assert exercise_4_arithmetic(10) == 9
# assert exercise_4_arithmetic(332) == 299
# assert exercise_4_arithmetic(668841) == 667999

### Bài 5 — Mini challenge: chữ số **không tăng** (khó)

Viết `exercise_5_non_increasing(n)` trả về số **lớn nhất `<= n`** mà các chữ số **không tăng** (mỗi cặp liền kề `x >= y`), ví dụ `5531 → 5531`, `1234 → 1111`, `120 → 111`.

```python
exercise_5_non_increasing(5531) -> 5531
exercise_5_non_increasing(1234) -> 1111
exercise_5_non_increasing(120)  -> 111
exercise_5_non_increasing(0)    -> 0
```

Gợi ý: tìm chỗ **tăng** đầu tiên `d[i] < d[i+1]`; số lớn nhất giữ tiền tố `d[0..i]` rồi điền gì vào các vị trí sau? Kiểm tra bằng oracle liệt kê `combinations_with_replacement` (sắp theo thứ tự giảm).

In [39]:
def exercise_5_non_increasing(n: int) -> int:
    # TODO
    pass


# assert exercise_5_non_increasing(5531) == 5531
# assert exercise_5_non_increasing(1234) == 1111
# assert exercise_5_non_increasing(120) == 111
# assert exercise_5_non_increasing(0) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `list(str(1204))` là danh sách ký tự; so sánh ký tự theo thứ tự `'0' < ... < '9'`.
- Mức 2: `"9" > "10"` so ký tự đầu `'9'` với `'1'`; `int("0099")` bỏ số 0 đầu.
- Mức 3: dòng 1 `['1', '2', '0', '4']`; dòng 2 `True True 99`; dòng 3 `True`.

**Bài 2**
- Mức 1: các chữ số `2,3,3,2,2,1`; tìm cặp `d[i] > d[i+1]` đầu tiên.
- Mức 2: cặp đầu tiên giảm là `(3, 2)` ở `i = 2`; cụm `3,3` bắt đầu ở `j = 1`.
- Mức 3: hạ `d[1]` từ `3` thành `2`, điền `9` từ vị trí `2`: `2,2,9,9,9,9` → `229999`.

**Bài 3**
- Mức 1: `for i in range(len(s) - 1, 0, -1)`.
- Mức 2: `if s[i - 1] > s[i]: s[i - 1] = str(int(s[i - 1]) - 1); mark = i`.
- Mức 3: `for k in range(mark, len(s)): s[k] = "9"`; `return int("".join(s))`.

**Bài 4**
- Mức 1: tìm `j` = chỉ số lớn nhất có `s[j] > s[j+1]` (bằng vòng lặp qua mọi cặp).
- Mức 2: `m = len(s) - 1 - j`; `n = (n // 10**m) * 10**m - 1`.
- Mức 3: nếu không có chỗ giảm thì `return n`; ngược lại lặp lại.

**Bài 5**
- Mức 1: tìm chỗ **tăng** đầu tiên `d[i] < d[i+1]`; nếu không có thì đáp án là `n`.
- Mức 2: số mới giữ tiền tố `d[0..i]` nhưng ở vị trí `i+1` không thể là `d[i+1]` (lớn hơn `d[i]`), nên ở đó chọn gì để vẫn nhỏ hơn `n` và không tăng?
- Mức 3: điền `d[i]` vào mọi vị trí sau `i`: `int(s[:i + 1] + s[i] * (len(s) - i - 1))`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
print(list(str(1204)))
print("3" > "2", "9" > "10", int("0099"))
print(all(a <= b for a, b in zip("1224", "224")))

assert list(str(1204)) == ["1", "2", "0", "4"]
assert ("3" > "2", "9" > "10", int("0099")) == (True, True, 99)
assert all(a <= b for a, b in zip("1224", "224")) is True
print("Bài 1 OK")

['1', '2', '0', '4']
True True 99
True
Bài 1 OK


### Đáp án Bài 2

`233221`: chữ số `2,3,3,2,2,1`. Cặp giảm đầu tiên là `(d[2], d[3]) = (3, 2)` nên `i = 2`. Cụm chữ số bằng `3` là `d[1..2]`, đầu cụm `j = 1`. Hạ `d[1]` thành `2` và điền `9` từ vị trí `2`: `2 2 9 9 9 9` = `229999`.

In [41]:
n2 = 233221
s2 = str(n2)
first_drop = next(i for i in range(len(s2) - 1) if s2[i] > s2[i + 1])
run_start = first_drop
while run_start > 0 and s2[run_start - 1] == s2[first_drop]:
    run_start -= 1
ans2 = Solution().monotoneIncreasingDigits(n2)
print(first_drop, run_start, ans2)

assert (first_drop, run_start, ans2) == (2, 1, 229999)
assert ans2 == oracle(n2)
print("Bài 2 OK")

2 1 229999
Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(n: int) -> int:
    s = list(str(n))
    mark = len(s)
    for i in range(len(s) - 1, 0, -1):
        if s[i - 1] > s[i]:
            s[i - 1] = str(int(s[i - 1]) - 1)
            mark = i
    for k in range(mark, len(s)):
        s[k] = "9"
    return int("".join(s))


assert exercise_3_solution(10) == 9
assert exercise_3_solution(1234) == 1234
assert exercise_3_solution(332) == 299
for n in range(0, 5000):
    assert exercise_3_solution(n) == oracle(n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(n: int) -> int:
    while True:
        s = str(n)
        j = -1
        for t in range(len(s) - 1):
            if s[t] > s[t + 1]:
                j = t
        if j == -1:
            return n
        m = len(s) - 1 - j
        n = (n // 10 ** m) * 10 ** m - 1


assert exercise_4_solution(10) == 9
assert exercise_4_solution(332) == 299
assert exercise_4_solution(668841) == 667999
rng = random.Random(4)
for _ in range(3000):
    n = rng.randint(0, 10 ** 9)
    assert exercise_4_solution(n) == oracle(n), n
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(n: int) -> int:
    s = str(n)
    for i in range(len(s) - 1):
        if s[i] < s[i + 1]:
            return int(s[:i + 1] + s[i] * (len(s) - i - 1))
    return n


non_increasing = sorted(
    int("".join(c[::-1]))
    for L in range(1, 11)
    for c in combinations_with_replacement("0123456789", L)
)
non_increasing = sorted(set(non_increasing))


def oracle_non_increasing(n):
    return non_increasing[bisect_right(non_increasing, n) - 1]


assert exercise_5_solution(5531) == 5531
assert exercise_5_solution(1234) == 1111
assert exercise_5_solution(120) == 111
assert exercise_5_solution(0) == 0
for n in range(0, 30_000):
    assert exercise_5_solution(n) == oracle_non_increasing(n), n
rng = random.Random(5)
for _ in range(5000):
    n = rng.randint(0, 10 ** 9)
    assert exercise_5_solution(n) == oracle_non_increasing(n), n
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def monotoneIncreasingDigits(self, n: int) -> int:
        s = list(str(n))
        mark = len(s)                                   # từ mark trở đi điền '9'
        for i in range(len(s) - 1, 0, -1):              # quét từ phải sang trái
            if s[i - 1] > s[i]:
                s[i - 1] = str(int(s[i - 1]) - 1)       # hạ chữ số bên trái đi 1
                mark = i
        for k in range(mark, len(s)):
            s[k] = "9"
        return int("".join(s))
```

Câu thần chú: **"gặp chữ số trái lớn hơn chữ số phải thì hạ chữ số trái đi 1 và đánh dấu; cuối cùng điền `9` từ chỗ đánh dấu."**

### Checklist trước khi submit

- [ ] Quét **từ phải sang trái** để việc hạ chữ số tự lan sang cụm chữ số bằng nhau.
- [ ] Điều kiện vi phạm là `s[i-1] > s[i]` (không phải `>=`): chữ số bằng nhau được phép.
- [ ] Có bước **điền `9`** từ `mark` đến hết.
- [ ] Trả về `int(...)`, không phải chuỗi (xử lý `"09"` của `n = 10`).
- [ ] Test: `0`, một chữ số, `10`, `11`, `100`, `332`, `1234`, `668841`, `10^9`.
- [ ] Không lùi từng đơn vị.
- [ ] Skeleton đúng: `class Solution`, method `monotoneIncreasingDigits(self, n: int) -> int`.